# Sparse Autoencoders for Feature Discovery

This notebook demonstrates how to train and use Sparse Autoencoders (SAEs) to discover interpretable features in world model activations.

## Overview
- Create and configure a Sparse Autoencoder
- Train the SAE on model activations
- Analyze discovered features
- Visualize sparse feature representations

In [ ]:
# Setup and imports
import torch
import numpy as np
from world_model_lens.sae.sparse_autoencoder import SparseAutoencoder
from world_model_lens import HookedWorldModel
from world_model_lens.utils.data_utils import collect_activations
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

torch.manual_seed(42)
np.random.seed(42)

print("SAE environment setup complete")

## Create Sparse Autoencoder

Initialize an SAE with the appropriate dimensions for the model's activation space.

In [ ]:
# Define SAE configuration
from world_model_lens.sae.sparse_autoencoder import SAEConfig

sae_config = SAEConfig(
    input_dim=512,           # Model activation dimension
    hidden_dim=2048,         # SAE hidden dimension (4x input)
    sparsity_coefficient=3e-4,  # L1 sparsity penalty
    learning_rate=1e-3,
    batch_size=256,
    dead_feature_threshold=1e-7,
    feature_sampling_steps=1000
)

# Initialize the SAE
sae = SparseAutoencoder(sae_config)

print(f"SAE created with:")
print(f"  Input dim: {sae.input_dim}")
print(f"  Hidden dim (features): {sae.hidden_dim}")
print(f"  Sparsity coefficient: {sae_config.sparsity_coefficient}")

## Collect Training Data

Gather activations from the world model by running inference on diverse inputs.

In [ ]:
# Load model
model = HookedWorldModel.from_pretrained("dreamerv3 Atari")
model.eval()

# Collect activations from a dataset
from world_model_lens.utils.data_utils import RandomDataset

dataset = RandomDataset(
    num_samples=10000,
    observation_dim=2048,
    action_dim=18
)

# Hook into the transition model to collect activations
activations = collect_activations(
    model=model,
    dataset=dataset,
    layer_name="transition.rssm.hidden",
    batch_size=512
)

print(f"Collected {len(activations)} activation vectors")
print(f"Activation shape: {activations.shape}")

## Train the SAE

Train the sparse autoencoder to reconstruct activations while learning sparse, interpretable features.

In [ ]:
# Prepare data loader
from torch.utils.data import DataLoader, TensorDataset

activations_tensor = torch.FloatTensor(activations)
train_dataset = TensorDataset(activations_tensor)
train_loader = DataLoader(train_dataset, batch_size=sae_config.batch_size, shuffle=True)

# Training loop
training_log = sae.train(
    train_loader,
    num_steps=10000,
    log_every=1000,
    eval_every=2000
)

print("\nTraining complete!")
print(f"Final reconstruction error: {training_log['final_recon_error']:.6f}")
print(f"Final sparsity: {training_log['final_sparsity']:.4f}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Reconstruction loss
ax1 = axes[0]
ax1.plot(training_log["recon_loss"], color="#3498db", linewidth=2)
ax1.set_xlabel("Step")
ax1.set_ylabel("Reconstruction Loss")
ax1.set_title("Reconstruction Loss")
ax1.grid(True, alpha=0.3)

# Sparsity loss
ax2 = axes[1]
ax2.plot(training_log["sparsity_loss"], color="#e67e22", linewidth=2)
ax2.set_xlabel("Step")
ax2.set_ylabel("Sparsity Loss")
ax2.set_title("Sparsity Loss")
ax2.grid(True, alpha=0.3)

# Feature activation distribution
ax3 = axes[2]
ax3.hist(training_log["feature_activations"], bins=50, color="#9b59b6", edgecolor="white")
ax3.set_xlabel("Feature Activation")
ax3.set_ylabel("Count")
ax3.set_title("Feature Activation Distribution")
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("sae_training_curves.png", dpi=150)
plt.show()

## Analyze Discovered Features

Examine the learned features to understand what concepts they represent.

In [ ]:
# Analyze feature properties
feature_stats = sae.analyze_features(activations_tensor)

print("Feature Analysis Summary:")
print("="*50)
print(f"Total features: {feature_stats['num_features']}")
print(f"Active features (>1% max): {feature_stats['num_active']}")
print(f"Dead features (<threshold): {feature_stats['num_dead']}")
print(f"Mean activation: {feature_stats['mean_activation']:.4f}")
print(f"Feature importance entropy: {feature_stats['importance_entropy']:.4f}")

In [ ]:
# Identify the most important features
top_features = sae.get_top_features(activations_tensor, k=20)

print("\nTop 20 Most Frequently Activated Features:")
print("-"*40)
for i, (feature_idx, activation_count) in enumerate(top_features, 1):
    print(f"  {i:2d}. Feature {feature_idx:4d}: {activation_count:6.0f} activations")

In [ ]:
# Find semantically similar features using cosine similarity
similar_features = sae.find_similar_features(
    target_feature_idx=42,
    activations=activations_tensor,
    top_k=5
)

print(f"Features most similar to Feature 42:")
for idx, similarity in similar_features:
    print(f"  Feature {idx}: similarity = {similarity:.4f}")

## Visualize Sparse Features

Create visualizations to understand the sparse feature space.

In [ ]:
# Visualize feature importance distribution
feature_importances = sae.compute_feature_importance(activations_tensor)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Feature importance histogram
ax1 = axes[0]
ax1.bar(range(len(feature_importances)), feature_importances, color="#2ecc71", alpha=0.7)
ax1.set_xlabel("Feature Index")
ax1.set_ylabel("Importance Score")
ax1.set_title("Feature Importance Scores")
ax1.axhline(y=np.mean(feature_importances), color="red", linestyle="--", label="Mean")
ax1.legend()

# Top features bar chart
ax2 = axes[1]
top_n = 20
top_indices = np.argsort(feature_importances)[-top_n:]
top_values = feature_importances[top_indices]
ax2.barh(range(top_n), top_values, color="#3498db")
ax2.set_yticks(range(top_n))
ax2.set_yticklabels([f"Feature {i}" for i in top_indices])
ax2.set_xlabel("Importance Score")
ax2.set_title(f"Top {top_n} Most Important Features")
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()

In [ ]:
# PCA visualization of feature space
# Project activations into SAE feature space
with torch.no_grad():
    feature_activations = sae.encode(activations_tensor[:1000])

# Apply PCA for visualization
pca = PCA(n_components=2)
projected = pca.fit_transform(feature_activations.numpy())

# Color by sparsity level
sparsity_per_sample = (feature_activations > 0.1).float().sum(dim=1).numpy()

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    projected[:, 0], projected[:, 1],
    c=sparsity_per_sample,
    cmap="viridis",
    alpha=0.6,
    s=20
)
plt.colorbar(scatter, label="Active Features Count")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.title("SAE Feature Space (PCA Projection)")
plt.savefig("feature_space_pca.png", dpi=150)
plt.show()

## Save the Trained SAE

Export the SAE for later use in analysis and patching.

In [ ]:
# Save SAE checkpoint
sae.save("trained_sae.pt")

# Also save feature interpretations
import json

feature_analysis = {
    "num_features": int(feature_stats['num_features']),
    "num_active": int(feature_stats['num_active']),
    "num_dead": int(feature_stats['num_dead']),
    "top_features": [(int(idx), float(cnt)) for idx, cnt in top_features[:10]],
    "importance_entropy": float(feature_stats['importance_entropy'])
}

with open("feature_analysis.json", "w") as f:
    json.dump(feature_analysis, f, indent=2)

print("SAE and analysis results saved!")
print("  - trained_sae.pt: SAE model weights")
print("  - feature_analysis.json: Feature statistics")